In [ ]:
from gpt4all import GPT4All

In [ ]:
model = GPT4All("orca-mini-3b-gguf2-q4_0")


In [ ]:
def build_prompt(job_title, matched_skills, missing_skills, similarity):
    return f"""
You are an AI helping explain resume-job matching.

Job Title: {job_title}

Matched Skills: {matched_skills}
Missing Skills: {missing_skills}
Similarity Score: {round(similarity * 100, 2)}%

Explain in 2–3 simple sentences why this resume matches or does not match the job.
"""


In [ ]:
def generate_explanation(prompt):
    response = model.generate(
        prompt,
        max_tokens=120,
        temp=0.3
    )
    return response.strip()


In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("../data/evaluation_results.csv")
df.head()

In [ ]:
df["llm_explanation"] = df.get("llm_explanation", "")

for idx, row in df.iterrows():
    
    # Skip if explanation already exists
    if isinstance(row["llm_explanation"], str) and row["llm_explanation"].strip():
        continue

    prompt = build_prompt(
        row["job_id"],
        row["skill_match_percent"],
        row["missing_skills"],
        row["similarity_score"]
    )

    try:
        explanation = generate_explanation(prompt)
        df.at[idx, "llm_explanation"] = explanation
        print(f"Saved explanation for index {idx}")

    except Exception as e:
        print(f"Error at index {idx}: {e}")
        df.at[idx, "llm_explanation"] = "LLM generation failed"


In [ ]:
df.to_csv("../data/final_results_with_explanations.csv",index=False)
